<a href="https://colab.research.google.com/github/Gabo190594/recommender-system-supermarket/blob/main/notebooks/07_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clonar tu repositorio desde GitHub
!git clone https://github.com/Gabo190594/recommender-system-supermarket.git

# Entrar a la carpeta del proyecto
%cd recommender-system-supermarket


Cloning into 'recommender-system-supermarket'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 86 (delta 36), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 776.10 KiB | 4.36 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/recommender-system-supermarket


In [3]:
from IPython.display import Markdown, display

display(Markdown("""
# 07. Evaluación del sistema

## Objetivo
Medir el desempeño del recomendador y compararlo con una estrategia base.

## ¿Qué hace este notebook?
1. Construye un baseline simple basado en popularidad.
2. Define una métrica básica de evaluación, como Hit Rate@K.
3. Evalúa si los productos recomendados coinciden con productos relevantes del usuario.
4. Resume el resultado obtenido.

## ¿Por qué evaluar?
La evaluación permite verificar si el sistema de recomendación aprende patrones útiles
o si solo replica comportamientos generales del dataset.

## Resultado esperado
Contar con una medida cuantitativa del desempeño del sistema.
"""))


# 07. Evaluación del sistema

## Objetivo
Medir el desempeño del recomendador y compararlo con una estrategia base.

## ¿Qué hace este notebook?
1. Construye un baseline simple basado en popularidad.
2. Define una métrica básica de evaluación, como Hit Rate@K.
3. Evalúa si los productos recomendados coinciden con productos relevantes del usuario.
4. Resume el resultado obtenido.

## ¿Por qué evaluar?
La evaluación permite verificar si el sistema de recomendación aprende patrones útiles
o si solo replica comportamientos generales del dataset.

## Resultado esperado
Contar con una medida cuantitativa del desempeño del sistema.


In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Markdown, display

# =========================================================
# 1. CARGA
# =========================================================
df = pd.read_csv("data/processed/reward_data.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

display(Markdown("""
# 07. Evaluación mejorada de sistemas de recomendación

## Objetivo
Comparar varios enfoques de recomendación para identificar cuál funciona mejor en este dataset.

## Modelos evaluados
1. Popularidad global
2. Popularidad por categoría favorita del usuario
3. User-based collaborative filtering
4. Item-based collaborative filtering
5. Modelo híbrido (item-based + popularidad)

## Métrica
Se utiliza **Hit Rate@5**.
"""))

# =========================================================
# 2. TRAIN / TEST SPLIT POR USUARIO
#    Última interacción -> test
# =========================================================
train_parts = []
test_parts = []

for user_id, user_data in df.groupby("user_id"):
    user_data = user_data.sort_values("timestamp")

    if len(user_data) < 2:
        continue

    train_parts.append(user_data.iloc[:-1])
    test_parts.append(user_data.iloc[-1:])

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

display(Markdown(f"""
## División de datos

- Registros train: **{len(train_df):,}**
- Registros test: **{len(test_df):,}**
- Usuarios evaluados: **{test_df['user_id'].nunique()}**
"""))

# =========================================================
# 3. MATRICES
# =========================================================
user_item = train_df.pivot_table(
    index="user_id",
    columns="product_id",
    values="reward_final",
    aggfunc="mean",
    fill_value=0
)

item_user = user_item.T

# similitud usuario-usuario
user_sim = cosine_similarity(user_item)
user_sim_df = pd.DataFrame(user_sim, index=user_item.index, columns=user_item.index)

# similitud item-item
item_sim = cosine_similarity(item_user)
item_sim_df = pd.DataFrame(item_sim, index=item_user.index, columns=item_user.index)

print("Shape user-item:", user_item.shape)
print("Shape item-item:", item_sim_df.shape)

# =========================================================
# 4. BASELINE: POPULARIDAD GLOBAL
# =========================================================
popular_products = (
    train_df.groupby("product_id")["reward_final"]
    .mean()
    .reset_index()
    .sort_values("reward_final", ascending=False)
)

def recommend_popularity(user_id=None, top_n=5):
    return popular_products["product_id"].head(top_n).tolist()

# =========================================================
# 5. POPULARIDAD POR CATEGORÍA FAVORITA
# =========================================================
user_fav_category = (
    train_df.groupby("user_id")["category"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    .to_dict()
)

category_popularity = (
    train_df.groupby(["category", "product_id"])["reward_final"]
    .mean()
    .reset_index()
    .sort_values(["category", "reward_final"], ascending=[True, False])
)

def recommend_category_popularity(user_id, top_n=5):
    if user_id not in user_fav_category:
        return recommend_popularity(top_n=top_n)

    fav_cat = user_fav_category[user_id]
    seen = set(train_df[train_df["user_id"] == user_id]["product_id"].unique())

    recs = category_popularity[category_popularity["category"] == fav_cat]
    recs = recs[~recs["product_id"].isin(seen)]

    if recs.empty:
        return recommend_popularity(top_n=top_n)

    return recs["product_id"].head(top_n).tolist()

# =========================================================
# 6. USER-BASED CF
# =========================================================
def recommend_user_based(user_id, top_n=5, min_sim=0.0):
    if user_id not in user_item.index:
        return []

    seen = set(train_df[train_df["user_id"] == user_id]["product_id"].unique())
    similar_users = user_sim_df[user_id].sort_values(ascending=False).drop(user_id, errors="ignore")

    scores = {}

    for sim_user, sim_score in similar_users.items():
        if sim_score <= min_sim:
            continue

        sim_user_data = train_df[train_df["user_id"] == sim_user][["product_id", "reward_final"]]

        for _, row in sim_user_data.iterrows():
            product_id = row["product_id"]
            if product_id in seen:
                continue

            scores[product_id] = scores.get(product_id, 0) + sim_score * row["reward_final"]

    if not scores:
        return []

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [product for product, _ in ranked[:top_n]]

# =========================================================
# 7. ITEM-BASED CF
# =========================================================
def recommend_item_based(user_id, top_n=5):
    if user_id not in user_item.index:
        return []

    user_history = train_df[train_df["user_id"] == user_id][["product_id", "reward_final"]]
    seen = set(user_history["product_id"].unique())

    scores = {}

    for _, row in user_history.iterrows():
        item_i = row["product_id"]
        reward_i = row["reward_final"]

        if item_i not in item_sim_df.index:
            continue

        similar_items = item_sim_df[item_i].sort_values(ascending=False)

        for item_j, sim_score in similar_items.items():
            if item_j == item_i:
                continue
            if item_j in seen:
                continue
            if sim_score <= 0:
                continue

            scores[item_j] = scores.get(item_j, 0) + sim_score * reward_i

    if not scores:
        return []

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [product for product, _ in ranked[:top_n]]

# =========================================================
# 8. HÍBRIDO: ITEM-BASED + POPULARIDAD
# =========================================================
popular_score_map = popular_products.set_index("product_id")["reward_final"].to_dict()

def recommend_hybrid(user_id, top_n=5, alpha=0.7):
    item_recs = recommend_item_based(user_id, top_n=50)
    seen = set(train_df[train_df["user_id"] == user_id]["product_id"].unique())

    candidate_products = set(item_recs) | set(popular_products["product_id"].head(100).tolist())
    candidate_products = [p for p in candidate_products if p not in seen]

    # reconstruir score item-based
    user_history = train_df[train_df["user_id"] == user_id][["product_id", "reward_final"]]
    item_scores = {}

    for _, row in user_history.iterrows():
        item_i = row["product_id"]
        reward_i = row["reward_final"]

        if item_i not in item_sim_df.index:
            continue

        similar_items = item_sim_df[item_i]

        for item_j in candidate_products:
            if item_j == item_i:
                continue
            sim_score = similar_items.get(item_j, 0)
            if sim_score > 0:
                item_scores[item_j] = item_scores.get(item_j, 0) + sim_score * reward_i

    final_scores = {}
    for p in candidate_products:
        s_item = item_scores.get(p, 0)
        s_pop = popular_score_map.get(p, 0)
        final_scores[p] = alpha * s_item + (1 - alpha) * s_pop

    if not final_scores:
        return recommend_popularity(top_n=top_n)

    ranked = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    return [product for product, _ in ranked[:top_n]]

# =========================================================
# 9. MÉTRICA
# =========================================================
def hit_at_k(real_products, recommended_products, k=5):
    recommended_topk = recommended_products[:k]
    return int(any(p in recommended_topk for p in real_products))

# =========================================================
# 10. EVALUACIÓN DE TODOS LOS MODELOS
# =========================================================
results = []

for _, row in test_df.iterrows():
    user_id = row["user_id"]
    real_product = row["product_id"]

    rec_pop = recommend_popularity(top_n=5)
    rec_cat = recommend_category_popularity(user_id, top_n=5)
    rec_user = recommend_user_based(user_id, top_n=5)
    rec_item = recommend_item_based(user_id, top_n=5)
    rec_hybrid = recommend_hybrid(user_id, top_n=5, alpha=0.7)

    results.append({
        "user_id": user_id,
        "real_product": real_product,
        "pop_hit": hit_at_k([real_product], rec_pop, 5),
        "cat_hit": hit_at_k([real_product], rec_cat, 5),
        "user_hit": hit_at_k([real_product], rec_user, 5),
        "item_hit": hit_at_k([real_product], rec_item, 5),
        "hybrid_hit": hit_at_k([real_product], rec_hybrid, 5),
    })

results_df = pd.DataFrame(results)
display(Markdown("## Resultados por usuario"))
display(results_df.head())

# =========================================================
# 11. RESUMEN COMPARATIVO
# =========================================================
comparison_df = pd.DataFrame({
    "Modelo": [
        "Popularidad",
        "Popularidad por categoría",
        "User-based",
        "Item-based",
        "Híbrido"
    ],
    "Hit Rate@5": [
        results_df["pop_hit"].mean(),
        results_df["cat_hit"].mean(),
        results_df["user_hit"].mean(),
        results_df["item_hit"].mean(),
        results_df["hybrid_hit"].mean(),
    ]
}).sort_values("Hit Rate@5", ascending=False).reset_index(drop=True)

display(Markdown("## Comparación de modelos"))
display(comparison_df)

# =========================================================
# 12. ANÁLISIS DINÁMICO
# =========================================================
best_model = comparison_df.iloc[0]["Modelo"]
best_score = comparison_df.iloc[0]["Hit Rate@5"]
worst_model = comparison_df.iloc[-1]["Modelo"]
worst_score = comparison_df.iloc[-1]["Hit Rate@5"]

second_score = comparison_df.iloc[1]["Hit Rate@5"]
gap = best_score - second_score

if gap >= 0.05:
    performance_text = "presenta una ventaja clara sobre los demás enfoques"
elif gap > 0:
    performance_text = "supera ligeramente al resto de modelos"
else:
    performance_text = "empata con otro enfoque"

display(Markdown(f"""
## Análisis dinámico de resultados

- Mejor modelo: **{best_model}**
- Hit Rate@5 del mejor modelo: **{best_score:.4f}**
- Peor modelo: **{worst_model}**
- Hit Rate@5 del peor modelo: **{worst_score:.4f}**
- Diferencia entre el primer y segundo modelo: **{gap:.4f}**

### Interpretación
El modelo **{best_model}** {performance_text}.
"""))

# =========================================================
# 13. CONCLUSIONES DINÁMICAS
# =========================================================
interaction_matrix = train_df.pivot_table(
    index="user_id",
    columns="product_id",
    values="reward_final"
)

num_total = interaction_matrix.shape[0] * interaction_matrix.shape[1]
num_observed = np.count_nonzero(~interaction_matrix.isna())
sparsity = 1 - (num_observed / num_total)

if sparsity > 0.90:
    sparsity_text = "alta sparsity, lo que dificulta especialmente los enfoques basados en usuarios."
else:
    sparsity_text = "una densidad suficiente para que los modelos colaborativos funcionen mejor."

if best_model in ["Item-based", "Híbrido"]:
    final_text = "Esto sugiere que, para este dataset, conviene explotar relaciones entre productos más que entre usuarios."
elif best_model == "Popularidad":
    final_text = "Esto sugiere que el dataset todavía no contiene suficiente estructura para que los modelos personalizados superen a una estrategia simple."
elif best_model == "Popularidad por categoría":
    final_text = "Esto sugiere que una personalización sencilla basada en preferencias generales del usuario ya aporta mejora sobre el baseline."
else:
    final_text = "Esto sugiere que el patrón de similitud entre usuarios sí logra capturar parte de las preferencias."

display(Markdown(f"""
## Conclusiones finales

- La evaluación se realizó usando una separación temporal en **train** y **test**.
- La matriz usuario-producto presenta una sparsity de **{sparsity:.4f}**, lo que indica {sparsity_text}
- El mejor resultado fue obtenido por **{best_model}** con un Hit Rate@5 de **{best_score:.4f}**.
- {final_text}
"""))


# 07. Evaluación mejorada de sistemas de recomendación

## Objetivo
Comparar varios enfoques de recomendación para identificar cuál funciona mejor en este dataset.

## Modelos evaluados
1. Popularidad global
2. Popularidad por categoría favorita del usuario
3. User-based collaborative filtering
4. Item-based collaborative filtering
5. Modelo híbrido (item-based + popularidad)

## Métrica
Se utiliza **Hit Rate@5**.



## División de datos

- Registros train: **14,499**
- Registros test: **500**
- Usuarios evaluados: **500**


Shape user-item: (500, 800)
Shape item-item: (800, 800)


## Resultados por usuario

,user_id,real_product,pop_hit,cat_hit,user_hit,item_hit,hybrid_hit
0,1,624,0,0,0,0,0
1,2,502,0,0,0,0,0
2,3,787,0,0,0,0,0
3,4,715,0,0,0,0,0
4,5,295,0,0,0,0,0


## Comparación de modelos

,Modelo,Hit Rate@5
0,Popularidad,0.010
1,Popularidad por categoría,0.010
2,Item-based,0.010
3,User-based,0.008
4,Híbrido,0.008



## Análisis dinámico de resultados

- Mejor modelo: **Popularidad**
- Hit Rate@5 del mejor modelo: **0.0100**
- Peor modelo: **Híbrido**
- Hit Rate@5 del peor modelo: **0.0080**
- Diferencia entre el primer y segundo modelo: **0.0000**

### Interpretación
El modelo **Popularidad** empata con otro enfoque.



## Conclusiones finales

- La evaluación se realizó usando una separación temporal en **train** y **test**.
- La matriz usuario-producto presenta una sparsity de **0.9656**, lo que indica alta sparsity, lo que dificulta especialmente los enfoques basados en usuarios.
- El mejor resultado fue obtenido por **Popularidad** con un Hit Rate@5 de **0.0100**.
- Esto sugiere que el dataset todavía no contiene suficiente estructura para que los modelos personalizados superen a una estrategia simple.


In [10]:
display(Markdown(f"""
# Resumen ejecutivo

El proceso de evaluación comparó distintos enfoques de recomendación sobre el dataset del proyecto.
El mejor desempeño fue obtenido por **{best_model}** con un Hit Rate@5 de **{best_score:.4f}**.

Aunque se identificó un modelo ganador, los resultados muestran que el nivel general de desempeño aún es bajo,
lo que evidencia que la principal oportunidad de mejora no está solo en el algoritmo,
sino en la calidad y densidad de los datos utilizados.

Desde una perspectiva de negocio, esto significa que el proyecto sí aporta valor como prueba de concepto,
pero todavía requiere madurez de datos antes de evolucionar hacia un recomendador personalizado de mayor complejidad.
"""))


# Resumen ejecutivo

El proceso de evaluación comparó distintos enfoques de recomendación sobre el dataset del proyecto.
El mejor desempeño fue obtenido por **Popularidad** con un Hit Rate@5 de **0.0100**.

Aunque se identificó un modelo ganador, los resultados muestran que el nivel general de desempeño aún es bajo,
lo que evidencia que la principal oportunidad de mejora no está solo en el algoritmo,
sino en la calidad y densidad de los datos utilizados.

Desde una perspectiva de negocio, esto significa que el proyecto sí aporta valor como prueba de concepto,
pero todavía requiere madurez de datos antes de evolucionar hacia un recomendador personalizado de mayor complejidad.
